In [6]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Literal, Optional, Tuple, Union

import numpy as np
import xarray as xr
from pathlib import Path

from ExperimentUtil import (
    run_dict,
    open_var_from_files,
    get_comp_dir,
    find_nc_files,
    collect_variable_list,
    flatten_variable_dict,
    open_compvar,
    set_monthly_time_to_midmonth_cftime
)

In [7]:
@dataclass
class SubseasonalWindMetricsConfig:
    # Domain for tropical verification
    lat_range: Tuple[float, float] = (-20.0, 20.0)
    lon_range: Optional[Tuple[float, float]] = None  # e.g., (0, 360) or None for full

    # Verification window relative to init (inclusive)
    lead_days_max: int = 120

    # Smoothing of daily metric curves
    metric_smooth_days: int = 7

    # Anomaly method
    anom_method: AnomMethod = "bandpass_20_100"

    # Bandpass settings for daily data (days)
    bp_low: int = 20
    bp_high: int = 100
    bp_numtaps: int = 201  # must be odd; larger = sharper but more edge loss

    # Running-mean removal settings (if rm60)
    rm_window: int = 60

    # Window means for bar chart
    lead_windows: Tuple[Tuple[int, int], ...] = ((5, 14), (15, 28), (29, 42))

    # Block bootstrap
    bootstrap_block_len: int = 7
    bootstrap_n: int = 2000
    bootstrap_ci: Tuple[float, float] = (2.5, 97.5)  # percentile CI

    # Variable names expected inside datasets
    # (you can map before passing in; class assumes these exist)
    varlist: Tuple[str, ...] = ("U850", "U200")  # extend to V850/V200 if you want

    # Coordinate names (adapt if needed)
    time_name: str = "time"
    lat_name: str = "lat"
    lon_name: str = "lon"


class SubseasonalWindMetrics:
    """
    Generates daily ACC/RMSE vs lag metrics for tropical winds/circulation.

    Inputs should already be on the *same grid* (regridded) and share lat/lon names
    (or be renamed before passing in). This class focuses on metrics output.

    Typical usage:
        m = SubseasonalWindMetrics(cfg, init_time="2012-01-01T00:00:00")
        out = m.compute_metrics(model_ds, era5_ds, exp_name="DARTBL")
        out.to_netcdf("metrics_DARTBL.nc")
    """

    def __init__(self, cfg: SubseasonalWindMetricsConfig, init_time: Union[str, np.datetime64]):
        self.cfg = cfg
        self.init_time = np.datetime64(init_time)

        if cfg.anom_method == "bandpass_20_100":
            if cfg.bp_numtaps % 2 == 0:
                raise ValueError("bp_numtaps must be odd for symmetric FIR (Lanczos) filter.")

    # ---------------------------
    # Public API
    # ---------------------------

    def compute_metrics(
        self,
        model_ds: xr.Dataset,
        era5_ds: xr.Dataset,
        exp_name: str,
        make_daily: bool = True,
    ) -> xr.Dataset:
        """
        Returns an xr.Dataset containing:
          - daily ACC(var, lag)
          - daily RMSE(var, lag)
          - smoothed ACC/RMSE (7-day by default)
          - window mean ACC + bootstrap CI for each lead window

        Assumptions:
          - model_ds and era5_ds overlap in time at least for [init, init+lead_days_max]
          - both datasets contain cfg.varlist as DataArrays with dims including time, lat, lon
        """
        cfg = self.cfg

        # 1) Daily mean (optional)
        if make_daily:
            model = self._to_daily(model_ds)
            era5 = self._to_daily(era5_ds)
        else:
            model, era5 = model_ds, era5_ds

        # 2) Subset time to [init, init+max]
        t0 = self.init_time
        t1 = t0 + np.timedelta64(cfg.lead_days_max, "D")
        model = self._subset_time(model, t0, t1)
        era5 = self._subset_time(era5, t0, t1)

        # 3) Subset spatial domain
        model = self._subset_domain(model)
        era5 = self._subset_domain(era5)

        # 4) Compute anomalies
        model_a = self._compute_anoms(model)
        era5_a = self._compute_anoms(era5)

        # 5) Align exactly on time
        model_a, era5_a = xr.align(model_a, era5_a, join="inner")

        # 6) Create lag coordinate in days since init
        lag = self._lag_days(model_a[cfg.time_name].values, t0)

        # 7) Compute daily ACC + RMSE
        acc = []
        rmse = []
        for v in cfg.varlist:
            a = self._daily_spatial_acc(model_a[v], era5_a[v])
            r = self._daily_area_rmse(model_a[v], era5_a[v])
            acc.append(a.assign_coords(var=v))
            rmse.append(r.assign_coords(var=v))

        acc_da = xr.concat(acc, dim="var").assign_coords(lag=("time", lag)).swap_dims({"time": "lag"})
        rmse_da = xr.concat(rmse, dim="var").assign_coords(lag=("time", lag)).swap_dims({"time": "lag"})

        # 8) Smooth daily curves for readability
        acc_sm = self._running_mean(acc_da, cfg.metric_smooth_days)
        rmse_sm = self._running_mean(rmse_da, cfg.metric_smooth_days)

        # 9) Window means + bootstrap CI (on ACC by default)
        win_stats = self._window_acc_bootstrap(acc_da)

        out = xr.Dataset(
            data_vars=dict(
                ACC=acc_da,
                RMSE=rmse_da,
                ACC_smooth=acc_sm,
                RMSE_smooth=rmse_sm,
                ACC_window_mean=win_stats["mean"],
                ACC_window_ci_low=win_stats["ci_low"],
                ACC_window_ci_high=win_stats["ci_high"],
            ),
            attrs=dict(
                exp_name=exp_name,
                init_time=str(t0),
                anom_method=cfg.anom_method,
                lat_range=str(cfg.lat_range),
                lon_range=str(cfg.lon_range),
                lead_days_max=str(cfg.lead_days_max),
            ),
        )
        return out

    # ---------------------------
    # Daily / subset utilities
    # ---------------------------

    def _to_daily(self, ds: xr.Dataset) -> xr.Dataset:
        """Convert time series to daily means if sub-daily; if already daily, returns unchanged."""
        cfg = self.cfg
        t = ds[cfg.time_name]
        # crude heuristic: if median dt < 1 day -> resample
        dt = np.median(np.diff(t.values).astype("timedelta64[h]").astype(int)) if t.size > 1 else 24
        if dt < 24:
            return ds.resample({cfg.time_name: "1D"}).mean()
        return ds

    def _subset_time(self, ds: xr.Dataset, t0: np.datetime64, t1: np.datetime64) -> xr.Dataset:
        cfg = self.cfg
        return ds.sel({cfg.time_name: slice(t0, t1)})

    def _subset_domain(self, ds: xr.Dataset) -> xr.Dataset:
        cfg = self.cfg
        lat = cfg.lat_name
        lon = cfg.lon_name

        ds2 = ds.sel({lat: slice(cfg.lat_range[0], cfg.lat_range[1])})
        if cfg.lon_range is not None:
            lo0, lo1 = cfg.lon_range
            # handle dateline-wrapping if needed (simple version)
            if lo0 <= lo1:
                ds2 = ds2.sel({lon: slice(lo0, lo1)})
            else:
                ds2 = xr.concat([ds2.sel({lon: slice(lo0, 360)}),
                                 ds2.sel({lon: slice(0, lo1)})], dim=lon)
        return ds2

    def _lag_days(self, time_vals: np.ndarray, init_time: np.datetime64) -> np.ndarray:
        d = (time_vals.astype("datetime64[D]") - init_time.astype("datetime64[D]")).astype(int)
        return d

    # ---------------------------
    # Anomalies
    # ---------------------------

    def _compute_anoms(self, ds: xr.Dataset) -> xr.Dataset:
        cfg = self.cfg
        if cfg.anom_method == "rm60":
            return self._rm_anoms(ds, cfg.rm_window)
        elif cfg.anom_method == "bandpass_20_100":
            return self._bandpass_anoms(ds, cfg.bp_low, cfg.bp_high, cfg.bp_numtaps)
        else:
            raise ValueError(f"Unknown anom_method: {cfg.anom_method}")

    def _rm_anoms(self, ds: xr.Dataset, window: int) -> xr.Dataset:
        """Anomaly = x - running_mean(x, window)."""
        cfg = self.cfg
        out = ds.copy()
        for v in cfg.varlist:
            x = ds[v]
            mu = x.rolling({cfg.time_name: window}, center=True, min_periods=max(5, window // 3)).mean()
            out[v] = x - mu
        return out

    def _bandpass_anoms(self, ds: xr.Dataset, low: int, high: int, numtaps: int) -> xr.Dataset:
        """
        Bandpass anomalies using a symmetric Lanczos FIR filter applied along time.
        This returns the bandpassed component directly (i.e., anomaly).
        """
        cfg = self.cfg
        out = ds.copy()
        for v in cfg.varlist:
            out[v] = self._lanczos_bandpass_1d(ds[v], low, high, numtaps)
        return out

    def _lanczos_bandpass_1d(self, da: xr.DataArray, low: int, high: int, numtaps: int) -> xr.DataArray:
        """
        Apply FIR bandpass to daily data along cfg.time_name.
        low/high in *days* for passband edges.
        """
        cfg = self.cfg
        tname = cfg.time_name

        # Build weights (daily sampling, fs = 1/day)
        w = self._lanczos_bandpass_weights(low=low, high=high, numtaps=numtaps)

        # Convolve along time with apply_ufunc (vectorized over lat/lon)
        # Uses 'same' length with NaNs at edges due to padding strategy.
        def _conv1d(x: np.ndarray, w: np.ndarray) -> np.ndarray:
            # x: (time,) or (..., time) depending on vectorization; we enforce time last
            # We'll assume apply_ufunc gives time as core dim.
            y = np.convolve(x, w, mode="same")
            # edge handling: mark edges as NaN where filter is not well-supported
            half = len(w) // 2
            y[:half] = np.nan
            y[-half:] = np.nan
            return y

        filtered = xr.apply_ufunc(
            _conv1d,
            da,
            xr.DataArray(w, dims=("kernel",)),
            input_core_dims=[[tname], ["kernel"]],
            output_core_dims=[[tname]],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[da.dtype],
        )
        return filtered

    def _lanczos_bandpass_weights(self, low: int, high: int, numtaps: int) -> np.ndarray:
        """
        Create Lanczos FIR bandpass weights for daily sampled data.
        low/high are periods in days.
        """
        if numtaps % 2 == 0:
            raise ValueError("numtaps must be odd.")

        # Convert periods (days) -> normalized cutoff frequencies (cycles/sample)
        # For daily sampling: sample rate fs = 1/day, Nyquist = 0.5 cycles/day.
        f1 = 1.0 / high  # lower frequency bound (cycles/day)
        f2 = 1.0 / low   # upper frequency bound (cycles/day)
        # normalized to Nyquist? For sinc design with cycles/sample, we can use f directly.

        M = numtaps
        n = np.arange(M) - (M - 1) / 2.0

        # Ideal bandpass = ideal lowpass(f2) - ideal lowpass(f1)
        def ideal_lowpass(fc: float, n: np.ndarray) -> np.ndarray:
            # fc in cycles/sample (cycles/day here, since sample=day)
            h = np.zeros_like(n, dtype=float)
            # sinc(2*fc*n) with np.sinc(x)=sin(pi x)/(pi x)
            h = 2 * fc * np.sinc(2 * fc * n)
            return h

        h = ideal_lowpass(f2, n) - ideal_lowpass(f1, n)

        # Lanczos window
        # lanczos(n) = sinc(n / a), where a = (M-1)/2
        a = (M - 1) / 2.0
        lanczos = np.sinc(n / a)
        w = h * lanczos

        # Normalize to unit gain in passband is tricky; for anomalies it’s often fine.
        # Enforce zero-mean (bandpass should already be ~zero-sum)
        w = w - w.mean()
        return w

    # ---------------------------
    # Metrics
    # ---------------------------

    def _coslat_weights(self, lat: xr.DataArray) -> xr.DataArray:
        """Cosine-lat area weights (normalized not required for correlation)."""
        return np.cos(np.deg2rad(lat))

    def _daily_spatial_acc(self, a: xr.DataArray, b: xr.DataArray) -> xr.DataArray:
        """
        Spatial anomaly correlation each day:
            ACC(t) = corr( a(t,lat,lon), b(t,lat,lon) ) with cos(lat) weighting.
        """
        cfg = self.cfg
        lat = cfg.lat_name

        wlat = self._coslat_weights(a[lat])
        # broadcast to lat/lon
        w = wlat / wlat.mean()

        # Weighted correlation:
        # corr = cov / (std_a * std_b), cov = mean_w( (a-mean_w(a))*(b-mean_w(b)) )
        def wmean(x):
            return (x * w).mean(dim=(lat, cfg.lon_name), skipna=True)

        am = wmean(a)
        bm = wmean(b)
        aa = a - am
        bb = b - bm

        cov = wmean(aa * bb)
        va = wmean(aa * aa)
        vb = wmean(bb * bb)
        acc = cov / np.sqrt(va * vb)
        acc.name = "ACC"
        return acc

    def _daily_area_rmse(self, a: xr.DataArray, b: xr.DataArray) -> xr.DataArray:
        """
        Area-weighted RMSE each day over domain.
        """
        cfg = self.cfg
        lat = cfg.lat_name
        lon = cfg.lon_name

        wlat = self._coslat_weights(a[lat])
        w2d = wlat / wlat.mean()

        mse = ((a - b) ** 2 * w2d).mean(dim=(lat, lon), skipna=True)
        rmse = np.sqrt(mse)
        rmse.name = "RMSE"
        return rmse

    def _running_mean(self, da: xr.DataArray, win: int) -> xr.DataArray:
        """Centered running mean along lag dimension (or time if lag not present)."""
        dim = "lag" if "lag" in da.dims else self.cfg.time_name
        return da.rolling({dim: win}, center=True, min_periods=max(3, win // 2)).mean()

    # ---------------------------
    # Window mean + block bootstrap
    # ---------------------------

    def _window_acc_bootstrap(self, acc: xr.DataArray) -> Dict[str, xr.DataArray]:
        """
        For each lead window, compute mean ACC and block-bootstrap CI along lag dimension.
        Returns dict of DataArrays with dims: (var, window)
        """
        cfg = self.cfg

        window_labels = [f"{a:02d}-{b:02d}" for a, b in cfg.lead_windows]
        means = []
        ci_lo = []
        ci_hi = []

        for (a, b), lab in zip(cfg.lead_windows, window_labels):
            seg = acc.sel(lag=slice(a, b))  # dims: var, lag
            m = seg.mean("lag", skipna=True)

            lo, hi = self._block_bootstrap_ci(seg, block_len=cfg.bootstrap_block_len,
                                             n=cfg.bootstrap_n, ci=cfg.bootstrap_ci)

            means.append(m.assign_coords(window=lab))
            ci_lo.append(lo.assign_coords(window=lab))
            ci_hi.append(hi.assign_coords(window=lab))

        mean_da = xr.concat(means, dim="window")
        lo_da = xr.concat(ci_lo, dim="window")
        hi_da = xr.concat(ci_hi, dim="window")
        return {"mean": mean_da, "ci_low": lo_da, "ci_high": hi_da}

    def _block_bootstrap_ci(
        self,
        seg: xr.DataArray,
        block_len: int,
        n: int,
        ci: Tuple[float, float],
    ) -> Tuple[xr.DataArray, xr.DataArray]:
        """
        Block bootstrap along lag dimension for seg(var, lag).
        Returns (ci_low(var), ci_high(var)).

        Note: with a single realization, this bootstrap quantifies sampling uncertainty
        within the lead window (not “between starts”).
        """
        rng = np.random.default_rng(12345)  # deterministic; make configurable if you want

        # bring into memory for bootstrap (window is small: e.g., 10-20 days)
        x = seg.transpose("var", "lag").values  # shape (nvar, T)
        nvar, T = x.shape

        # indices for blocks
        starts = np.arange(0, max(1, T - block_len + 1))
        if starts.size == 0:
            # too short; fallback to NaNs
            lo = np.full((nvar,), np.nan)
            hi = np.full((nvar,), np.nan)
            return (xr.DataArray(lo, dims=("var",), coords={"var": seg["var"]}),
                    xr.DataArray(hi, dims=("var",), coords={"var": seg["var"]}))

        boot_means = np.empty((n, nvar), dtype=float)

        for i in range(n):
            # sample blocks until we cover T
            idx = []
            while len(idx) < T:
                s = rng.choice(starts)
                idx.extend(range(s, s + block_len))
            idx = np.array(idx[:T])

            sample = x[:, idx]
            boot_means[i, :] = np.nanmean(sample, axis=1)

        lo_q, hi_q = np.percentile(boot_means, [ci[0], ci[1]], axis=0)
        lo_da = xr.DataArray(lo_q, dims=("var",), coords={"var": seg["var"]})
        hi_da = xr.DataArray(hi_q, dims=("var",), coords={"var": seg["var"]})
        return lo_da, hi_da


In [ ]:
AnomMethod = Literal["bandpass_20_100", "rm60"]
